In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (
    StandardScaler,
    Normalizer,
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    SplineTransformer,
)
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_extraction import FeatureHasher
from sklearn.compose import make_column_transformer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split, cross_val_score


In [ ]:
# Revisit the OKCupid data
df = pd.read_csv("../04_categorical/profiles_revised.csv")

In [ ]:
# Target: predict if job is stem related
df["job"].value_counts()
df["stem_job"] = df["job"].str.contains("computer|science", case=False, na=False)
df["stem_job"].value_counts() / len(df)
df.drop(columns="job", inplace=True)

In [ ]:
df.info()

In [ ]:
df.plot.hist(column="age", bins=50)
np.sum(df["age"] > 60)

## Understanding missing values
In addition to looking at how many of which feature are missing, it's useful to understand whether missing values tend to co-occur.

In [ ]:
# Store a copy before we mess with it
og_df = df.copy()

In [ ]:
# set income < 0 to missing
df.loc[df["income"] < 0, "income"] = np.nan
df.head()

In [ ]:
print("Missing values by percentage of dataset")
df.isna().sum().sort_values(ascending=False) / len(df) * 100

In [ ]:
# How many missing values per sample?
missing_per_sample = df.isna().sum(axis=1).value_counts()
plt.bar(missing_per_sample.index, missing_per_sample)
plt.xlabel("Number of missing features")
plt.ylabel("Number of samples")

In [ ]:
# Maybe a heatmap can show trends?
# number of missing per sample
missing_samples = df.isna().sum(axis=1).sort_values()
# drop with none missing and sort 
missing_samples.drop(missing_samples[missing_samples == 0].index, inplace=True)

# Features with samples missing
missing_features = df.isna().sum(axis=0).sort_values()
missing_features.drop(missing_features[missing_features == 0].index, inplace=True)

sns.heatmap(df.loc[missing_samples.index, missing_features.index].isna())

In [ ]:
df.loc[missing_samples.index, missing_features.index].isna()

In [ ]:
# how about correlations between missing values?
missing_cor = df.loc[missing_samples.index, missing_features.index].isna().corr()
# zero out the diagonal so we can visualize it better
missing_cor -= np.identity(missing_cor.shape[0])
sns.heatmap(missing_cor)

In [ ]:
# before looking at relationship to target, split the data
train, test = train_test_split(df, stratify=df["stem_job"], random_state=12345)

In [ ]:
# look at the ones that have a lot missing
train_missing = train.isna().sum(axis=0).sort_values(ascending=False)
# filter down to the > 20% missing
n_missing = train_missing[train_missing > 0.2 * len(train)]
print(n_missing)

# percentage in each target category
train_prop = train["stem_job"].value_counts() / len(train)
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)
for feat, ax in zip(n_missing.index, axes.flatten()):
    counts = train.groupby("stem_job")[feat].apply(lambda x: x.isna().sum())
    # normalize
    counts /= n_missing[feat]
    # compare to the overall proportion of stem/not stem
    ax.bar([-0.1, .9], train_prop, width=0.2, label="Overall")
    ax.bar([0.1, 1.1], counts, width=0.2, label=f"{feat}")
    ax.set_xticks([0, 1], labels=["Non-STEM", "STEM"])
    ax.set_title(f"Proportion of missing {feat}")
    ax.legend()

# Look at value_counts for the features with lots of missing data, grouped by stem_job


In [ ]:
# what about the non-missing values from categories with a lot missing?
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for feat, ax in zip(n_missing.index, axes.flatten()):
    # income is the only continuous one, do a box plot
    if feat == "income":
        train[["income", "stem_job"]].dropna().plot.box(column="income", ax=ax,by="stem_job")
        ax.set_xlabel("STEM job")
        ax.set_xticks([1,2], labels=["Non-STEM", "STEM"])
        ax.set_yscale("log")
    else:
        # categorical needs value_counts
        # Thanks to https://stackoverflow.com/questions/48238305/bar-plot-with-groupby for unstack(0) magic
        feat_cat_counts = train[[feat, "stem_job"]].dropna().groupby("stem_job")[feat].value_counts().unstack(0)
        
        # normalize by total number in each stem category
        feat_cat_counts /= feat_cat_counts.sum()
        # plot magic
        feat_cat_counts.plot.bar(ax=ax)


## Dealing with missing values

In [ ]:
# select features to use in the model
numeric_features = ["age", "height", "income"]
cat_features = ["drinks", "education", "sex","pets","religion","offspring"]

In [ ]:
for cat in cat_features:
    print(cat,"cardinality:")
    print(len(train[cat].value_counts()))

In [ ]:
# Define the trickier encoders
drink_enc = OrdinalEncoder(
            categories=[
                [
                    "not at all",
                    "rarely",
                    "socially",
                    "often",
                    "very often",
                    "desperately",
                ]
            ],
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )

# This took a while mucking around to figure out the right magic between df/series
def split_edu(df):
    edu = df["education"].copy()
    edu[edu.isna()] = ""
    return edu.str.split(" ")

# poorly named, now using for two different features
edu_enc = make_pipeline(
        FunctionTransformer(split_edu, validate=False),
        FeatureHasher(n_features=8, input_type="string"),
    )

def split_religion(df):
    edu = df["religion"].copy()
    edu[edu.isna()] = ""
    return edu.str.split(" ")

# poorly named, now using for two different features
religion_enc = make_pipeline(
        FunctionTransformer(split_religion, validate=False),
        FeatureHasher(n_features=16, input_type="string"),
    )


Iterations
1. Initial guess (same as 05_numeric/numericdemo.ipynb): array([0.68783959, 0.67537844, 0.68029499, 0.69439772, 0.6787424 ])
2. Adding on pets, offspring, religion: array([0.69434752, 0.68200837, 0.73779637, 0.69177127, 0.72365666])
3. increase religion feature hash size to 16: array([0.71598046, 0.67991632, 0.67782427, 0.73361227, 0.78087927])
4. Log transform the income: array([0.65903025, 0.69461744, 0.24354982, 0.4006895 , 0.67467467]) -> bad idea, keep the power transformer

In [ ]:
# Build the preprocessing pipeline

numeric_pipeline = make_pipeline(
    SimpleImputer(strategy="mean", add_indicator=True),
    # StandardScaler(),
    PowerTransformer(method="yeo-johnson"),
)

income_pipeline = make_pipeline(
    SimpleImputer(strategy="mean", add_indicator=True),
    FunctionTransformer(np.log1p),
)

preprocessor = make_column_transformer(
    (numeric_pipeline, ["age", "height"]),
    (income_pipeline, ["income"]),
    (OneHotEncoder(handle_unknown="ignore"), ["sex","pets","offspring"]),
    (drink_enc, ["drinks"]),
    (edu_enc, ["education"]),
    (religion_enc, ["religion"])

)
preprocessor

In [ ]:
pro_out = preprocessor.fit_transform(train)
plt.hist(pro_out[:,0].todense(), bins=50)

In [ ]:
# Now add on a model!
pipeline = make_pipeline(
    preprocessor,
    SGDClassifier(class_weight="balanced"), #class_weight="balanced" does some magic
)

cross_val_score(pipeline, train, train["stem_job"])

## Extra plots used in slides

In [ ]:
# assume -1 is encoding missingness
plt.hist(og_df[og_df["income"] > 0]["income"], bins=50)
plt.xlabel("Income > 0 in OKCupid Dataset")
plt.ylabel("Number of instances")
plt.savefig("../../static/img/06-income-hist.png")

In [ ]:
# assume -1 is encoding missingness
log_inc = np.log(og_df[og_df["income"] > 0]["income"])
plt.hist(log_inc, bins=50)
plt.xlabel("Income > 0 in OKCupid Dataset")
plt.ylabel("Log number of instances")
plt.savefig("../../static/img/06-income-hist-log.png")

In [ ]:
# What about boxplots vs STEM?
# Only use the training data since we're looking at relationship with predictor
plot_df = train[train["income"] > 0]
fig, ax = plt.subplots()
plot_df.plot.box(column="income", ax=ax,by="stem_job")
ax.set_xlabel("STEM job")
ax.set_xticks([1,2], labels=["Non-STEM", "STEM"])
ax.set_ylabel("Income (> 0)")
plt.yscale("log")
plt.savefig("../../static/img/06-income-boxplot.png")

In [ ]:
# Those outliers are all overlapping, try spreading them out with jitter
fig, ax = plt.subplots()
plot_df.query("income < 140000").plot.box(column="income", ax=ax,by="stem_job")
ax.set_xlabel("STEM job")
ax.set_xticks([1,2], labels=["Non-STEM", "STEM"])
ax.set_ylabel("Income (> 0)")
plt.yscale("log")

# we need a random number generator
rng = np.random.default_rng(seed=67)
outliers = plot_df.query("income >= 140000")
jit_x = rng.normal(size=outliers.shape[0], scale=0.1) + 1 + outliers["stem_job"] * 1
jit_y = rng.normal(size=outliers.shape[0], scale=10000) + outliers["income"] 
plt.scatter(jit_x, jit_y, alpha=0.3)
